# Matrix mechanics: Heisenberg's oscillator from ladder operators

*quantum · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/matrix_mechanics)

No wavefunctions, no grid: in the number basis $|0\rangle, \dots, |N-1\rangle$ the lowering operator is the matrix $a_{mn} = \sqrt n\,\delta_{m,n-1}$, and $X$, $P$ and $H = \tfrac12(P^2 + X^2)$ follow by matrix products. Its eigenvalues are $n + \tfrac12$ and $[X, P] = i$. With torch.linalg.matrix_exp the operator $X(t) = U^\dagger X U$ moves (Heisenberg picture) or the state $U|\alpha\rangle$ moves (Schrödinger picture); both give the same $\langle X\rangle(t)$, and a coherent state circles phase space like a classical mass on a spring.

**Vector form**

$$X = \frac{a + a^\dagger}{\sqrt2},\quad P = \frac{i(a^\dagger - a)}{\sqrt2},\quad H = \tfrac12(P^2 + X^2) + \lambda X^4,\quad [X, P] = i,\quad X(t) = e^{iHt}\,X\,e^{-iHt}$$

**Index (tensor) form**

$$a_{mn} = \sqrt n\,\delta_{m,n-1},\qquad H_{mn} = \tfrac12\big(P_{mk}P_{kn} + X_{mk}X_{kn}\big),\qquad \langle X\rangle(t) = c^*_m\,U^*_{km}X_{kl}U_{ln}\,c_n = \big(U_{km}c_m\big)^*X_{kl}\big(U_{ln}c_n\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $a_{mn}$ | `lowering(N)` | (N, N) | $\sqrt1, \sqrt2, \dots$ on the superdiagonal: $a/n\rangle = \sqrt n\,/n-1\rangle$ |
| $X_{mn},\ P_{mn}$ | `position(a), momentum(a)` | (N, N), (N, N) | $(a + a^\dagger)/\sqrt2$ and $i(a^\dagger - a)/\sqrt2$, both Hermitian |
| $H_{mn}$ | `0.5*(einsum('mk,kn->mn', P, P) + einsum('mk,kn->mn', X, X))` | (N, N) | repeated $k$ ⇒ summed: matrix products; equals $a^\dagger a + \tfrac12$ except the last entry |
| $[X, P]_{mn}$ | `X @ P - P @ X` | (N, N) | $i\,\delta_{mn}$, but $i(1 - N)$ in the corner: a finite matrix cannot have $[X, P] = i$ (its trace is 0) |
| $E_n$ | `torch.linalg.eigvalsh(H)` | (N,) | $n + \tfrac12$; with $\lambda X^4$ shifted by $\approx \lambda\,\tfrac34(2n^2 + 2n + 1)$ |
| $U_{mn}(t)$ | `torch.linalg.matrix_exp(-1j * t * H)` | (T, N, N) | time-evolution operator $e^{-iHt}$ for all times at once |
| $X(t)_{mn}$ | `einsum('tkm,kl,tln->tmn', U.conj(), X, U)` | (T, N, N) | repeated $k, l$ ⇒ summed: Heisenberg picture, the operator moves |
| $c_n$ | `coherent_state(alpha, N)` | (N,) | $e^{-/\alpha/^2/2}\alpha^n/\sqrt{n!}$: a displaced ground state |
| $\langle X\rangle(t)$ | `einsum('tm,mn,tn->t', c_t.conj(), X, c_t)` | (T,) | Schrödinger picture with $c(t) = Uc$: the same numbers as the Heisenberg picture |

Units $\hbar = m = \omega = 1$. This is quantum mechanics as Heisenberg, Born and Jordan first wrote it in 1925: observables are matrices and their products do not commute. Truncating to $N$ states is the only approximation. It breaks $[X, P] = i$ in the last diagonal entry, which must equal $i(1 - N)$ because the trace of a commutator of finite matrices is 0, and it gives the last basis state the energy $(N-1)/2$ instead of $N - \tfrac12$; states well below $N$ are exact. In the Heisenberg picture each entry $X(t)_{mn} = X_{mn}e^{i(E_m - E_n)t}$ rotates at a Bohr frequency, so for the oscillator $X(t) = X\cos t + P\sin t$, the classical solution written for matrices. The coherent state $|\alpha\rangle$ has $\langle X\rangle(t) = \sqrt2\,\mathrm{Re}(\alpha e^{-it})$ and circles phase space with radius $\sqrt2\,\alpha$. The grid calculation in Quantum harmonic oscillator gives the same ladder and the same swing from a completely different representation. Try $\lambda > 0$: the levels are no longer evenly spaced, the Bohr frequencies spread, and the circle spirals inwards as the packet dephases. For small $\lambda$ the energies match first-order perturbation theory, $\lambda\langle n|X^4|n\rangle = \lambda\,\tfrac34(2n^2 + 2n + 1)$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/matrix_mechanics/main.py
"""Main loop: parameters → matrices → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import commutator, dagger, hamiltonian, number_hamiltonian


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    N, alpha, lam = params["n_states"], params["alpha"], params["lam"]
    a = tensors.lowering(N, dev)  # (N, N)
    X, P = tensors.position(a), tensors.momentum(a)  # (N, N) each
    H = hamiltonian(X, P, lam)  # (N, N)
    C = commutator(X, P)  # (N, N) ≈ i·I

    E = solve.energies(H)
    E0 = solve.energies(hamiltonian(X, P))  # λ = 0 ladder, for the console table
    exact_H = number_hamiltonian(a)
    print(f"N = {N} states.  H Hermitian: {torch.allclose(H, dagger(H))}")
    print(
        f"lambda = 0:  max |1/2(P^2 + X^2) - (a^dag a + 1/2)| over the top-left {N - 1}x{N - 1} block = "
        f"{(hamiltonian(X, P) - exact_H)[:-1, :-1].abs().max().item():.1e}"
    )
    print(f"[X, P] diagonal / i: {C.diagonal().imag[:3].tolist()} ... last = {C[-1, -1].imag.item():.1f} (= 1 - N)")
    n = torch.arange(N, dtype=torch.float64, device=dev)
    pert = solve.first_order_energies(n, lam)
    for k in range(4):
        print(
            f"E_{k} = {E[k].item():9.5f}   harmonic {E0[k].item():.5f} (n + 1/2 = {k + 0.5})   "
            f"1st-order PT {pert[k].item():9.5f}"
        )
    print(
        f"truncation artefact: last diagonal entry of 1/2(P^2 + X^2) = {hamiltonian(X, P).real[-1, -1].item():.1f}"
        f" = (N - 1)/2 instead of N - 1/2 = {exact_H.real[-1, -1].item():.1f}"
    )

    times = torch.linspace(0.0, params["periods"] * 2 * math.pi, 241, dtype=torch.float64, device=dev)
    U = solve.propagators(H, times)  # (T, N, N)
    c = tensors.coherent_state(complex(alpha, 0.0), N, dev)  # (N,)
    ops = {"X": X, "P": P}
    heis, moving = solve.heisenberg_picture(U, ops, c)
    schr = solve.schrodinger_picture(U, ops, c)
    gap = max((heis[k] - schr[k]).abs().max().item() for k in ops)
    classical = math.sqrt(2) * alpha * torch.cos(times)
    print(f"max |<X>_Heisenberg - <X>_Schrodinger| = {gap:.1e}   (the two pictures agree)")
    print(f"max |<X> - sqrt(2) Re(alpha e^-it)| = {(heis['X'] - classical).abs().max().item():.1e}")
    return plot.animate(times, moving["X"], C, heis, schr, alpha, lam, out_dir / "matrix_mechanics.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/matrix_mechanics/equation.py
r"""Physics → tensor translation.

Heisenberg's matrix mechanics for the oscillator (units ħ = m = ω = 1) in the number basis |0⟩ … |N−1⟩:

    a_{mn} = \sqrt{n}\,\delta_{m,n-1}                       (lowering operator)
    X = (a + a^\dagger)/\sqrt2,     P = i(a^\dagger - a)/\sqrt2
    H_{mn} = \tfrac12 (P_{mk}P_{kn} + X_{mk}X_{kn}) + \lambda X^4_{mn}      (k summed)
    [X, P]_{mn} = X_{mk}P_{kn} - P_{mk}X_{kn} = i\,\delta_{mn}  (except the last entry: truncation)

Time evolution, U = e^{-iHt}, in two equivalent pictures:

    Heisenberg:   X(t)_{mn} = U^*_{km} X_{kl} U_{ln},     ⟨X⟩ = c^*_m X(t)_{mn} c_n       (k, l, m, n summed)
    Schrödinger:  c_m(t) = U_{mn} c_n,                     ⟨X⟩ = c^*_m(t) X_{mn} c_n(t)
"""

import torch


def dagger(A: torch.Tensor) -> torch.Tensor:
    r"""(A^\dagger)_{mn} = A^*_{nm}.   (..., N, N) → (..., N, N)"""
    return A.conj().transpose(-2, -1)


def matmul(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    r"""(AB)_{mn} = A_{mk} B_{kn}  (k summed).   (N, N), (N, N) → (N, N)"""
    return torch.einsum("mk,kn->mn", A, B)


def commutator(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    r"""[A, B] = AB - BA.   (N, N) → (N, N)"""
    return matmul(A, B) - matmul(B, A)


def hamiltonian(X: torch.Tensor, P: torch.Tensor, lam: float = 0.0) -> torch.Tensor:
    r"""H = \tfrac12(P^2 + X^2) + \lambda X^4.   (N, N) → (N, N) Hermitian"""
    X2 = matmul(X, X)
    return 0.5 * (matmul(P, P) + X2) + lam * matmul(X2, X2)


def number_hamiltonian(a: torch.Tensor) -> torch.Tensor:
    r"""The exact oscillator H = a^\dagger a + \tfrac12 = diag(n + ½).   (N, N) → (N, N)"""
    eye = torch.eye(len(a), dtype=a.dtype, device=a.device)
    return matmul(dagger(a), a) + 0.5 * eye


def heisenberg(U: torch.Tensor, op: torch.Tensor) -> torch.Tensor:
    r"""O(t)_{mn} = U^*_{km} O_{kl} U_{ln}  (k, l summed).   U: (T, N, N), op: (N, N) → (T, N, N)"""
    return torch.einsum("tkm,kl,tln->tmn", U.conj(), op, U)


def schrodinger(U: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    r"""c_m(t) = U_{mn} c_n  (n summed).   U: (T, N, N), c: (N,) → (T, N)"""
    return torch.einsum("tmn,n->tm", U, c)


def expectation(c: torch.Tensor, op: torch.Tensor) -> torch.Tensor:
    r"""⟨O⟩ = c^*_m O_{mn} c_n  (m, n summed).   c: (..., N), op: (..., N, N) → (...) real"""
    return torch.einsum("...m,...mn,...n->...", c.conj(), op, c).real


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/matrix_mechanics/tensors.py
"""Build the concrete matrices in the number basis |0⟩ … |N−1⟩ and the coherent start state."""

import math

import torch


def lowering(N: int, device: torch.device) -> torch.Tensor:
    """a_{mn} = √n δ_{m,n−1}: √1, √2, …, √(N−1) on the first superdiagonal. Shape (N, N) complex."""
    root_n = torch.sqrt(torch.arange(1, N, dtype=torch.float64, device=device))  # (N-1,)
    return torch.diag(root_n, 1).to(torch.complex128)


def position(a: torch.Tensor) -> torch.Tensor:
    """X = (a + a†)/√2. Shape (N, N), real symmetric (stored complex)."""
    return (a + a.conj().T) / math.sqrt(2)


def momentum(a: torch.Tensor) -> torch.Tensor:
    """P = i(a† − a)/√2. Shape (N, N), Hermitian with imaginary entries."""
    return 1j * (a.conj().T - a) / math.sqrt(2)


def coherent_state(alpha: complex, N: int, device: torch.device) -> torch.Tensor:
    """|α⟩: c_n = e^{−|α|²/2} αⁿ/√(n!), the minimum-uncertainty packet centred on x = √2 Re α. Shape (N,) complex.

    Computed through logarithms so αⁿ and n! never overflow; the tail beyond N is cut off.
    """
    n = torch.arange(N, dtype=torch.float64, device=device)  # (N,)
    if alpha == 0:
        return (n == 0).to(torch.complex128)  # |0⟩, the ground state
    log_mag = -(abs(alpha) ** 2) / 2 + n * math.log(abs(alpha)) - 0.5 * torch.lgamma(n + 1)
    phase = torch.exp(1j * n * math.atan2(alpha.imag, alpha.real))  # (N,)
    return torch.exp(log_mag) * phase


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/matrix_mechanics/solve.py
"""Diagonalise H, build U(t) = e^{−iHt} with matrix_exp, and evolve ⟨X⟩, ⟨P⟩ in both pictures."""

import torch

from .equation import expectation, heisenberg, schrodinger


def energies(H: torch.Tensor) -> torch.Tensor:
    """Eigenvalues of the Hermitian H, ascending: (N,)."""
    return torch.linalg.eigvalsh(H)


def propagators(H: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
    """U(t) = exp(−iHt) for every time at once (batched matrix exponential). (T,) → (T, N, N)"""
    return torch.linalg.matrix_exp(-1j * times[:, None, None].to(H.dtype) * H)


def heisenberg_picture(U: torch.Tensor, ops: dict, c: torch.Tensor) -> tuple[dict, dict]:
    """Operators move, the state stays: ⟨O⟩(t) = c† (U†OU) c.

    Returns ({name: ⟨O⟩ (T,)}, {name: O(t) (T, N, N)}).
    """
    moving = {name: heisenberg(U, op) for name, op in ops.items()}  # (T, N, N) each
    return {name: expectation(c, Ot) for name, Ot in moving.items()}, moving


def schrodinger_picture(U: torch.Tensor, ops: dict, c: torch.Tensor) -> dict:
    """The state moves, operators stay: ⟨O⟩(t) = c(t)† O c(t) with c(t) = U c. Returns {name: (T,)}."""
    ct = schrodinger(U, c)  # (T, N)
    return {name: expectation(ct, op) for name, op in ops.items()}


def first_order_energies(n: torch.Tensor, lam: float) -> torch.Tensor:
    """Perturbation theory for H₀ + λX⁴: E_n ≈ n + ½ + λ⟨n|X⁴|n⟩ = n + ½ + λ·¾(2n² + 2n + 1)."""
    return n + 0.5 + lam * 0.75 * (2 * n**2 + 2 * n + 1)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/matrix_mechanics/plot.py
"""Animate the Heisenberg-picture matrix X(t), the commutator, and ⟨X⟩, ⟨P⟩ of the coherent state."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import BoundaryNorm

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    times: torch.Tensor,
    X_t: torch.Tensor,
    C: torch.Tensor,
    heis: dict,
    schr: dict,
    alpha: float,
    lam: float,
    path: Path,
    block: int = 10,
) -> Path:
    t = times.cpu().numpy()
    Xb = X_t[:, :block, :block].real.cpu().numpy()  # (T, block, block) top-left corner of Re X(t)
    Ci = C.imag.cpu().numpy()  # (N, N): [X, P] / i
    xh, ph = heis["X"].cpu().numpy(), heis["P"].cpu().numpy()  # (T,)
    xs = schr["X"].cpu().numpy()  # (T,) the same numbers from the Schrödinger picture
    N = Ci.shape[0]
    r = math.sqrt(2) * alpha  # radius of the classical circle

    fig, axes = plt.subplots(2, 2, figsize=(8, 5.6))
    (ax_x, ax_c), (ax_ph, ax_t) = axes

    # Re X(t)_{mn}: the entries next to the diagonal rotate like e^{±it}
    vmax = np.abs(Xb).max()
    levels = np.linspace(-vmax, vmax, 12)
    img = ax_x.imshow(Xb[0], cmap="RdBu_r", norm=BoundaryNorm(levels, 256))
    ax_x.set_title(f"Re X(t) = Re U†XU   (first {block} states)", fontsize=9)
    ax_x.set_xlabel("n")
    ax_x.set_ylabel("m")
    ax_x.grid(False)
    ax_x.set_xticks(range(0, block, 3))
    ax_x.set_yticks(range(0, block, 3))
    cbar_x = fig.colorbar(img, ax=ax_x, shrink=0.85, ticks=[-vmax, 0, vmax], format="%.2f")
    cbar_x.ax.tick_params(labelsize=7)

    # [X, P]/i: the identity, except the bottom-right corner 1 − N
    c_levels = [1 - N - 0.5, -0.5, 0.5, 1.5]
    img_c = ax_c.imshow(Ci, cmap="PuOr", norm=BoundaryNorm(c_levels, 256))
    ax_c.set_title(f"[X, P] / i   (N = {N})", fontsize=9)
    ax_c.annotate(
        f"{1 - N}",
        (N - 1, N - 1),
        xytext=(0.45 * N, 0.85 * N),
        fontsize=8,
        arrowprops={"arrowstyle": "->"},
    )
    ax_c.set_xlabel("n")
    ax_c.grid(False)
    cbar = fig.colorbar(img_c, ax=ax_c, shrink=0.85, ticks=[1 - N, 0, 1])
    cbar.ax.tick_params(labelsize=7)

    # phase space: ⟨X⟩, ⟨P⟩ of the coherent state against the classical circle
    angle = np.linspace(0, 2 * np.pi, 200)
    ax_ph.plot(r * np.cos(angle), -r * np.sin(angle), "--", color="#9ca3af", lw=1, label="classical circle")
    (trail,) = ax_ph.plot([], [], color=ACCENT[0], lw=1.5, label="⟨X⟩, ⟨P⟩")
    (dot,) = ax_ph.plot([], [], "o", color=ACCENT[0], ms=7)
    lim = 1.2 * max(r, np.abs(xh).max(), np.abs(ph).max(), 0.5)
    ax_ph.set_xlim(-lim, lim)
    ax_ph.set_ylim(-lim, lim)
    ax_ph.set_aspect("equal")
    ax_ph.set_xlabel("⟨X⟩")
    ax_ph.set_ylabel("⟨P⟩")
    ax_ph.set_title(f"⟨X⟩, ⟨P⟩ of |α = {alpha:g}⟩,  λ = {lam:g}   (dashed: classical)", fontsize=9)

    # ⟨X⟩(t) from both pictures
    ax_t.plot(t, r * np.cos(t), "--", color="#9ca3af", lw=1, label="√2 Re(α e^(−it))")
    (line_h,) = ax_t.plot([], [], color=ACCENT[0], lw=2, label="Heisenberg: c† X(t) c")
    (dots_s,) = ax_t.plot([], [], "o", color=ACCENT[1], ms=3, label="Schrödinger: c(t)† X c(t)")
    ax_t.set_xlim(0, t[-1])
    ax_t.set_ylim(-1.5 * lim, 1.5 * lim)
    ax_t.set_xlabel("t")
    ax_t.set_title("⟨X⟩(t)", fontsize=9)
    ax_t.legend(loc="upper right", fontsize=7, ncol=1)
    clock = ax_t.text(0.02, 0.05, "", transform=ax_t.transAxes, family="monospace", fontsize=8)

    def update(step: int):
        img.set_data(Xb[step])
        trail.set_data(xh[: step + 1], ph[: step + 1])
        dot.set_data([xh[step]], [ph[step]])
        line_h.set_data(t[: step + 1], xh[: step + 1])
        dots_s.set_data(t[: step + 1 : 4], xs[: step + 1 : 4])
        clock.set_text(f"t = {t[step]:5.2f}   |Heis − Schr| = {abs(xh[step] - xs[step]):.0e}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 60), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'alpha': 2.0,
    'lam': 0.0,
    'periods': 3.0,
    'n_states': 40
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.matrix_mechanics")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.matrix_mechanics.main")
problem_dir = Path("problems/quantum/matrix_mechanics")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
